## Optimization Data Preparation

The objective of this notebook is to transform raw operational tables into optimization-ready structures.

Outputs from this notebook will be used directly in the linear programming model developed in Notebook 04.

In [63]:
# Import libraries for data analysis and visualization

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Display settings for easier dataframe review
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [64]:
# Load all tables from the supply chain workbook

file_path = "../data/raw/Supply chain logistics problem.xlsx"

order_list = pd.read_excel(file_path, sheet_name="OrderList")
freight_rates = pd.read_excel(file_path, sheet_name="FreightRates")
wh_costs = pd.read_excel(file_path, sheet_name="WhCosts")
wh_capacities = pd.read_excel(file_path, sheet_name="WhCapacities")
products_per_plant = pd.read_excel(file_path, sheet_name="ProductsPerPlant")
vmi_customers = pd.read_excel(file_path, sheet_name="VmiCustomers")
plant_ports = pd.read_excel(file_path, sheet_name="PlantPorts")

In [85]:
# Calculate total demand by customer

customer_demand = (
    order_list
    .groupby('Customer')['Unit quantity']
    .sum()
    .reset_index()
)

customer_demand.columns = [
    'Customer',
    'Demand'
]

customer_demand.head()

,Customer,Demand
0,V555555555555555555_17,266457
1,V555555555555555555_42,470632
2,V555555555555555555_45,116136
3,V555555555555555555_46,12080
4,V555555555555555_23,375


In [66]:
print("Customers:", len(customer_demand))
print("Total Demand:", customer_demand['Demand'].sum())

Customers: 46
Total Demand: 29513315


In [67]:
# Create plant capacity table

plant_capacity = (
    wh_capacities
    .copy()
)

plant_capacity.head()

,Plant ID,Daily Capacity
0,PLANT15,11
1,PLANT17,8
2,PLANT18,111
3,PLANT05,385
4,PLANT02,138


In [68]:
print("Plants:", len(plant_capacity))
print(
    "Total Capacity:",
    plant_capacity['Daily Capacity '].sum()
)

Plants: 19
Total Capacity: 5791


In [69]:
# Combine capacity and cost

plant_summary = (
    plant_capacity
    .merge(
        wh_costs,
        left_on='Plant ID',
        right_on='WH',
        how='left'
    )
)

plant_summary.head()

,Plant ID,Daily Capacity,WH,Cost/unit
0,PLANT15,11,PLANT15,1.415063
1,PLANT17,8,PLANT17,0.428947
2,PLANT18,111,PLANT18,2.036254
3,PLANT05,385,PLANT05,0.488144
4,PLANT02,138,PLANT02,0.477504


In [70]:
plant_summary

,Plant ID,Daily Capacity,WH,Cost/unit
0,PLANT15,11,PLANT15,1.415063
1,PLANT17,8,PLANT17,0.428947
2,PLANT18,111,PLANT18,2.036254
3,PLANT05,385,PLANT05,0.488144
4,PLANT02,138,PLANT02,0.477504
5,PLANT01,1070,PLANT01,0.566976
6,PLANT06,49,PLANT06,0.554088
7,PLANT10,118,PLANT10,0.493582
8,PLANT07,265,PLANT07,0.371424
9,PLANT14,549,PLANT14,0.634330


In [71]:
# Verify all plants matched successfully

print("Plants in capacity table:", plant_capacity['Plant ID'].nunique())
print("Plants in summary table:", plant_summary['Plant ID'].nunique())

Plants in capacity table: 19
Plants in summary table: 19


In [72]:
# Review final plant summary

plant_summary.sort_values(
    by='Daily Capacity ',
    ascending=False
)

,Plant ID,Daily Capacity,WH,Cost/unit
5,PLANT01,1070,PLANT01,0.566976
14,PLANT03,1013,PLANT03,0.517502
18,PLANT04,554,PLANT04,0.428503
9,PLANT14,549,PLANT14,0.634330
15,PLANT13,490,PLANT13,0.469707
10,PLANT16,457,PLANT16,1.919808
3,PLANT05,385,PLANT05,0.488144
12,PLANT11,332,PLANT11,0.555247
8,PLANT07,265,PLANT07,0.371424
11,PLANT12,209,PLANT12,0.773132


In [73]:
print(freight_rates.columns.tolist())
freight_rates.head()

['Carrier', 'orig_port_cd', 'dest_port_cd', 'minm_wgh_qty', 'max_wgh_qty', 'svc_cd', 'minimum cost', 'rate', 'mode_dsc', 'tpt_day_cnt', 'Carrier type']


,Carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum cost,rate,mode_dsc,tpt_day_cnt,Carrier type
0,V444_6,PORT08,PORT09,250.0,499.99,DTD,43.2272,0.7132,AIR,2,V88888888_0
1,V444_6,PORT08,PORT09,65.0,69.99,DTD,43.2272,0.7512,AIR,2,V88888888_0
2,V444_6,PORT08,PORT09,60.0,64.99,DTD,43.2272,0.7892,AIR,2,V88888888_0
3,V444_6,PORT08,PORT09,50.0,54.99,DTD,43.2272,0.8272,AIR,2,V88888888_0
4,V444_6,PORT08,PORT09,35.0,39.99,DTD,43.2272,1.0552,AIR,2,V88888888_0


In [74]:
# Summarize freight lanes

route_summary = (
    freight_rates.groupby(
        ['orig_port_cd', 'dest_port_cd']
    )['rate']
    .mean()
    .reset_index()
)

route_summary.head()

,orig_port_cd,dest_port_cd,rate
0,PORT02,PORT09,1.874696
1,PORT03,PORT09,9.979378
2,PORT04,PORT09,1.940713
3,PORT05,PORT09,2.872483
4,PORT06,PORT09,2.532242


In [75]:
# Calculate average transportation cost by route

route_summary = (
    freight_rates
    .groupby(
        ['orig_port_cd', 'dest_port_cd']
    )
    .agg({
        'rate': 'mean',
        'tpt_day_cnt': 'mean'
    })
    .reset_index()
)

route_summary = route_summary.rename(columns={
    'rate': 'avg_rate',
    'tpt_day_cnt': 'avg_transit_days'
})

route_summary.head()

,orig_port_cd,dest_port_cd,avg_rate,avg_transit_days
0,PORT02,PORT09,1.874696,1.619469
1,PORT03,PORT09,9.979378,2.911111
2,PORT04,PORT09,1.940713,1.584416
3,PORT05,PORT09,2.872483,1.269406
4,PORT06,PORT09,2.532242,1.989562


In [76]:
# Review route inventory

print("Routes:", len(route_summary))
print("Origin Ports:", route_summary['orig_port_cd'].nunique())
print("Destination Ports:", route_summary['dest_port_cd'].nunique())

Routes: 10
Origin Ports: 10
Destination Ports: 1


In [77]:
# Review plant-port relationships

plant_ports.head()

,Plant Code,Port
0,PLANT01,PORT01
1,PLANT01,PORT02
2,PLANT02,PORT03
3,PLANT03,PORT04
4,PLANT04,PORT05


In [78]:
# Count available port connections per plant

plant_port_summary = (
    plant_ports
    .groupby('Plant Code')
    .size()
    .reset_index(name='Available Ports')
)

plant_port_summary.head()

,Plant Code,Available Ports
0,PLANT01,2
1,PLANT02,1
2,PLANT03,1
3,PLANT04,1
4,PLANT05,1


In [79]:
# Review product production constraints

products_per_plant.head()

,Plant Code,Product ID
0,PLANT15,1698815
1,PLANT17,1664419
2,PLANT17,1664426
3,PLANT17,1672826
4,PLANT17,1674916


In [80]:
# Review columns

print(products_per_plant.columns.tolist())

['Plant Code', 'Product ID']


In [81]:
# Count products available at each plant

product_coverage = (
    products_per_plant
    .groupby('Plant Code')
    .size()
    .reset_index(name='Products Available')
    .sort_values(
        by='Products Available',
        ascending=False
    )
)

product_coverage.head(10)

,Plant Code,Products Available
3,PLANT03,781
1,PLANT01,220
13,PLANT13,150
4,PLANT04,134
5,PLANT05,127
10,PLANT10,121
2,PLANT02,116
16,PLANT16,113
11,PLANT11,96
12,PLANT12,57


In [82]:
optimization_inputs = {
    "customer_demand": customer_demand,
    "plant_capacity": plant_capacity,
    "warehouse_cost": wh_costs,
    "plant_summary": plant_summary,
    "route_summary": route_summary,
    "plant_ports": plant_ports,
    "products_per_plant": products_per_plant
}

In [83]:
for name, table in optimization_inputs.items():
    print(f"{name}: {table.shape}")

customer_demand: (46, 2)
plant_capacity: (19, 2)
warehouse_cost: (19, 2)
plant_summary: (19, 4)
route_summary: (10, 4)
plant_ports: (22, 2)
products_per_plant: (2036, 2)
